# INDUSMIND AI - Environment Setup & Dependency Scaffolding
### AI-Powered Industrial Knowledge Intelligence Platform

This notebook acts as the foundational environment builder for **INDUSMIND AI** development. It mounts Google Drive, checks for GPU acceleration telemetry, installs all required AI/OCR/RAG/Knowledge Graph libraries, maps out the project directory structure, configures logging, and exports a configuration settings JSON file to unify subsequent processing notebooks.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
import os

print("Mounting Google Drive...")
try:
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except Exception as e:
    print(f"Google Drive Mount Notice/Error: {e}")

In [ ]:
# 2. Detect GPU availability
import torch

print("Detecting hardware acceleration telemetry...")
gpu_available = torch.cuda.is_available()
device = "cuda" if gpu_available else "cpu"
print(f"GPU Available: {gpu_available}")
if gpu_available:
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Memory Alloc Limit: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: GPU acceleration not active. Deep learning inference runs on CPU.")

In [ ]:
# 3. Install System & AI dependencies
import sys

print("Installing system packages (Tesseract OCR binary)...")
!apt-get update -qq && apt-get install -y -qq tesseract-ocr libtesseract-dev
print("System packages installed successfully.")

print("Installing required AI, OCR, and Graph python modules...")
dependencies = [
    "transformers",
    "sentence-transformers",
    "langchain",
    "langchain-community",
    "faiss-cpu",
    "easyocr",
    "pytesseract",
    "spacy",
    "pdfplumber",
    "pymupdf",
    "networkx",
    "neo4j",
    "huggingface_hub"
]

# Install quietly to prevent long cell output streams
!pip install -q {" ".join(dependencies)}

print("Downloading English SpaCy pipeline...")
!python -m spacy download en_core_web_sm -q

print("SUCCESS: All dependencies and pipelines installed successfully.")

print("\n" + "="*70)
print("⚠️  CRITICAL ACTION REQUIRED:")
print("Go to the top menu: Click 'Runtime' -> 'Restart session' (or 'Restart runtime')")
print("This is required to reload the updated image processing libraries (Pillow)!")
print("="*70)

In [ ]:
# 4. Create Project Folders in Google Drive
import os

BASE_DIR = "/content/drive/MyDrive/INDUSMIND_AI"
subfolders = [
    "datasets",
    "models",
    "embeddings",
    "vector_db",
    "knowledge_graph",
    "outputs",
    "logs"
]

print(f"Creating project folders in Google Drive under: {BASE_DIR}")
for folder in subfolders:
    path = os.path.join(BASE_DIR, folder)
    os.makedirs(path, exist_ok=True)
    print(f"  [Created/Verified] -> {path}")

print("SUCCESS: Directory structures mapped successfully.")

In [ ]:
# 5. Configure logging
import logging
import sys

log_file_path = os.path.join(BASE_DIR, "logs", "setup.log")

# Configure logger to print to console and save to a file
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    handlers=[
        logging.FileHandler(log_file_path),
        logging.StreamHandler(sys.stdout)
    ]
)

logger = logging.getLogger("INDUSMIND_Setup")
logger.info("Environment configuration logged successfully.")

In [ ]:
# 6. Verify package versions (Crash-Resilient Audit)
import logging
logger = logging.getLogger("INDUSMIND_Setup")

logger.info("Auditing installed dependency versions:")

packages_to_verify = [
    ("PyTorch", "torch"),
    ("Transformers", "transformers"),
    ("Sentence Transformers", "sentence_transformers"),
    ("LangChain", "langchain"),
    ("Faiss", "faiss"),
    ("EasyOCR", "easyocr"),
    ("PyTesseract", "pytesseract"),
    ("spaCy", "spacy"),
    ("PDFPlumber", "pdfplumber"),
    ("PyMuPDF (fitz)", "fitz"),
    ("NetworkX", "networkx"),
    ("Neo4j Driver", "neo4j"),
    ("Hugging Face Hub", "huggingface_hub")
]

packages = {}
for label, module_name in packages_to_verify:
    try:
        mod = __import__(module_name)
        if label == "PyTesseract":
            try:
                version = mod.get_tesseract_version()
            except Exception:
                version = "Installed (binary check skipped)"
        elif label == "PyMuPDF (fitz)":
            version = mod.__version__
        else:
            version = getattr(mod, "__version__", "Installed")
        packages[label] = version
        logger.info(f"  ✅ {label}: {version}")
    except Exception as e:
        packages[label] = f"Failed to load: {e}"
        logger.warning(f"  ❌ {label}: import error ({e})")
        if "PIL" in str(e) or "_Ink" in str(e):
            logger.warning("     💡 HINT: Restart your Colab session (Runtime -> Restart session) to reload Pillow in memory!")

In [ ]:
# 7. Save configuration JSON file for later notebooks
import json
from datetime import datetime

config_path = os.path.join(BASE_DIR, "config.json")

config_data = {
    "project_name": "INDUSMIND AI",
    "base_directory": BASE_DIR,
    "gpu_accelerator": {
        "available": gpu_available,
        "device": torch.cuda.get_device_name(0) if gpu_available else "cpu"
    },
    "data_directories": {
        "datasets": os.path.join(BASE_DIR, "datasets"),
        "models": os.path.join(BASE_DIR, "models"),
        "embeddings": os.path.join(BASE_DIR, "embeddings"),
        "vector_db": os.path.join(BASE_DIR, "vector_db"),
        "knowledge_graph": os.path.join(BASE_DIR, "knowledge_graph"),
        "outputs": os.path.join(BASE_DIR, "outputs"),
        "logs": os.path.join(BASE_DIR, "logs")
    },
    "verified_versions": packages,
    "timestamp_initialized": datetime.utcnow().isoformat() + "Z"
}

try:
    with open(config_path, "w") as f:
        json.dump(config_data, f, indent=4)
    logger.info(f"SUCCESS: Config settings saved to: {config_path}")
except Exception as e:
    logger.error(f"Failed to save config.json: {e}")

## 🚀 Recommended AI Models for INDUSMIND AI

To achieve enterprise-grade accuracy inside Google Colab (especially using a free-tier **T4 GPU** with 15GB VRAM), we recommend the following state-of-the-art open-source models:

### 1. Document OCR & Layout Parsing
- **EasyOCR** (default): Best for multi-language detection, fast out-of-the-box.
- **Tesseract OCR**: Fast, rule-based text block parsing.
- **`microsoft/layoutlmv3-base`**: State-of-the-art document AI model for invoice, receipt, and technical drawing structured entity extraction.

### 2. Sentence Embeddings (Vector Database & RAG)
- **`BAAI/bge-large-en-v1.5`**: Leading open-source text embedder on the MTEB leaderboard. Highly accurate semantic matching.
- **`sentence-transformers/all-MiniLM-L6-v2`**: Extremely fast, lightweight, and low-memory embedding generator, perfect for prototyping.

### 3. Entity Extraction & Zero-Shot Classification
- **`ursul/gliner_medium-v2.1`**: Zero-shot Entity Extraction (GLiNER). Allows extracting any arbitrary custom entity (e.g., \"PLANT SITE\", \"BOILER TYPE\", \"TEMP LIMIT\") without fine-tuning!
- **spaCy (`en_core_web_sm`)**: Excellent rule-based pipeline matching.

### 4. Generative RAG & Copilot LLM
- **`microsoft/Phi-3-mini-4k-instruct`**: Outstanding 3.8B parameter model. Highly performant on T4 GPU (and even CPUs), excellent reasoning for its size.
- **Meta-Llama-3-8B-Instruct** (or similar Llama-3 class instruction model): State-of-the-art 8B parameter LLM for advanced conversation, report summary, and formatting output as JSON.

In [ ]:
# 8. Verify loading key AI modules & models
import sys

try:
    print("Verifying spaCy pipeline loading...")
    import spacy
    nlp = spacy.load("en_core_web_sm")
    doc = nlp('INDUSMIND AI is initialized successfully.')
    print(f"  ✅ spaCy Token Check: {[token.text for token in doc]}")
except Exception as e:
    print(f"  ❌ spaCy verification failed: {e}")
    print("     💡 HINT: Run '!python -m spacy download en_core_web_sm' first.")

try:
    print("\nVerifying embedding model downloading and caching...")
    from sentence_transformers import SentenceTransformer
    embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    embeddings = embedder.encode(['INDUSMIND AI telemetry data'])
    print(f"  ✅ SentenceTransformer Check: Embeddings generated successfully. Dimensions: {embeddings.shape[1]}")
except Exception as e:
    print(f"  ❌ SentenceTransformer verification failed: {e}")
    if "PIL" in str(e) or "_Ink" in str(e):
        print("     💡 HINT: Restart your Colab session (Runtime -> Restart session) to reload Pillow in memory!")
    else:
        print(f"     Details: {e}")